# TrimCI-Flow — Uncoupled Fragment Baseline

This notebook runs the current `uncoupled` solver on Fe4S4 and verifies the regression-locked determinant baseline.

**Scientific role:** establish the no-coupling fragment determinant count used as the comparison point for meanfield and future DMET.

**Known baseline:**
- `fragment_n_dets = [51, 51, 16]`
- `total_dets = 118`
- brute-force TrimCI reference = `10095` determinants
- fragment energies are informational only; they are not summable.

Run inside the `qflowenv` environment at `/home/unfunnypanda/Proj_Flow/qflowenv/`.


In [1]:
# Cell 1 — Environment and path setup
import sys, os

# Make sure the project root is on the path
PROJECT_ROOT = os.path.expanduser('~/Proj_Flow')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print(f'Working directory: {os.getcwd()}')
print(f'Python: {sys.version}')

Working directory: /home/unfunnypanda/Proj_Flow
Python: 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]


In [2]:
# Cell 2 — Imports
import json
import datetime
from pathlib import Path
import numpy as np

from TrimCI_Flow.uncoupled import run_fragmented_trimci
from TrimCI_Flow.core import determinant_summary

print('Imports OK')


Imports OK


In [3]:
# Cell 3 — Configuration
ROOT = Path.cwd()
FCIDUMP = ROOT / 'Fe4S4_251230orbital_-327.1920_10kdets' / \
    'Fe4S4_251230orbital_-327.1920_10kdets' / 'fcidump_cycle_6'
WINDOW_SIZE = 15
STRIDE = 10
OUTPUT_DIR = ROOT / 'TrimCI_Flow' / 'Outputs' / 'uncoupled' / 'outs_notebook_uncoupled'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'FCIDUMP: {FCIDUMP}')
print(f'window_size={WINDOW_SIZE}, stride={STRIDE}')
print(f'Output dir: {OUTPUT_DIR.relative_to(ROOT)}/')


FCIDUMP: /home/unfunnypanda/Proj_Flow/Fe4S4_251230orbital_-327.1920_10kdets/Fe4S4_251230orbital_-327.1920_10kdets/fcidump_cycle_6
window_size=15, stride=10
Output dir: TrimCI_Flow/Outputs/uncoupled/outs_notebook_uncoupled/


In [4]:
# Cell 4 — Run uncoupled baseline
# trimci_config=None uses the current DEFAULT_CONFIG:
#   threshold=0.06, max_final_dets='auto' (~51 for 15-orb frags),
#   max_rounds=2, num_runs=1, pool_build_strategy='heat_bath'

print(f'Starting uncoupled run at {datetime.datetime.now().isoformat()}')
result = run_fragmented_trimci(str(FCIDUMP), window_size=WINDOW_SIZE, stride=STRIDE)
print(f'Finished at {datetime.datetime.now().isoformat()}')


Starting uncoupled run at 2026-04-15T14:08:08.607526
[C++ Workflow] Starting with 1 initial determinants
[C++ Workflow] Total possible configurations: 32207175
[C++ Workflow] Sparsity detected: h1=1.000000, eri=1.000000
[C++ Workflow] Double exc table: 105 entries in 0.004981s
[C++] Iteration 1/200000
[PoolBuild] Starting pool build: target_size=100, threshold=0.06, mode=heat_bath, factor=1
[PoolBuild] target_size reached, stopping.
[PoolBuild] Final pool size: 191, final threshold: 0.06
[Trim] Start. Pool=191
[Trim] Core-core H precomputed: 1 dets, 1 nnz, 0.00670766s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=11
[Trim] Final diagonalization...
[Trim] Final E=-183.1948868662
[C++] Iteration 1 energy: -183.194887
[C++] Core set: 1, Raw dets: 11
[C++] Core set: 1 -> 10 (first cycle)
[C++] Iteration 1 time: 0.043902s (Total: 0.052194s)
[C++] Iteration 2/200000
[PoolBuild] Starting pool build: target_size=200, threshold=0.0600000000, mode=heat_bath, factor=1
[PoolBuild] threshold 

In [5]:
# Cell 5 — Display determinant summary
det_sum = determinant_summary(result)

TrimCI-Flow — Determinant Summary
  Fragment 0: orbs [ 2..33]    n_dets=    51    energy=   -187.7502 Ha
  Fragment 1: orbs [ 4..33]    n_dets=    51    energy=   -219.7179 Ha
  Fragment 2: orbs [ 0..35]    n_dets=    16    energy=   -247.6590 Ha
------------------------------------------------------------
  Total dets     :      118
  Brute-force ref:    10095  (E = -327.1920 Ha)
  Ratio          :    0.012x brute-force
  Savings        :   +98.8%  (negative = MORE dets than brute-force)


In [6]:
# Cell 6 — Regression check (must not fail)
EXPECTED_TOTAL = 118
EXPECTED_NDETS = [51, 51, 16]

assert result.total_dets == EXPECTED_TOTAL, (
    f'REGRESSION FAIL: total_dets={result.total_dets}, expected {EXPECTED_TOTAL}'
)
assert result.fragment_n_dets == EXPECTED_NDETS, (
    f'REGRESSION FAIL: fragment_n_dets={result.fragment_n_dets}, expected {EXPECTED_NDETS}'
)
print(f'REGRESSION PASS: total_dets={result.total_dets}, fragment_n_dets={result.fragment_n_dets}')

# Note: fragment energies vary run-to-run due to stochastic heat-bath sampling (num_runs=1).
# Only determinant counts are asserted; energies are informational.
print(f'Fragment energies (informational, stochastic): {result.fragment_energies}')

REGRESSION PASS: total_dets=118, fragment_n_dets=[51, 51, 16]
Fragment energies (informational, stochastic): [-187.75020040863893, -219.71786393946374, -247.6589841523237]


In [7]:
# Cell 7 — Save outputs
metadata = {
    'run_type': 'uncoupled_notebook_regression',
    'timestamp': datetime.datetime.now().isoformat(),
    'fcidump': str(FCIDUMP),
    'window_size': WINDOW_SIZE,
    'stride': STRIDE,
    'trimci_config': 'DEFAULT_CONFIG',
    'total_dets': result.total_dets,
    'brute_force_dets': result.brute_force_dets,
    'fragment_n_dets': result.fragment_n_dets,
    'fragment_energies': result.fragment_energies,
    'fragment_orbs': result.fragment_orbs,
    'ratio': result.total_dets / result.brute_force_dets,
    'savings_pct': (1 - result.total_dets / result.brute_force_dets) * 100,
    'regression_pass': True,
}

frag_details = [
    {'fragment_index': i, 'orbs': orbs, 'n_orbs': len(orbs),
     'n_dets': nd, 'energy_Ha': en}
    for i, (orbs, nd, en) in enumerate(
        zip(result.fragment_orbs, result.fragment_n_dets, result.fragment_energies)
    )
]

(OUTPUT_DIR / 'run_metadata.json').write_text(json.dumps(metadata, indent=2) + '\n')
(OUTPUT_DIR / 'fragment_results.json').write_text(json.dumps(frag_details, indent=2) + '\n')
(OUTPUT_DIR / 'determinant_summary.txt').write_text(
    f'fragment_n_dets = {result.fragment_n_dets}\n'
    f'total_dets = {result.total_dets}\n'
    f'brute_force_dets = {result.brute_force_dets}\n'
    f'ratio = {result.total_dets / result.brute_force_dets:.4f}\n'
    f'savings_pct = {(1 - result.total_dets / result.brute_force_dets)*100:.1f}%\n'
    f'fragment_energies = {result.fragment_energies}\n'
    f'regression_pass = True\n'
)

print('Saved:')
for fn in ['run_metadata.json', 'fragment_results.json', 'determinant_summary.txt']:
    print(f'  {(OUTPUT_DIR / fn).relative_to(ROOT)}')


Saved:
  TrimCI_Flow/Outputs/uncoupled/outs_notebook_uncoupled/run_metadata.json
  TrimCI_Flow/Outputs/uncoupled/outs_notebook_uncoupled/fragment_results.json
  TrimCI_Flow/Outputs/uncoupled/outs_notebook_uncoupled/determinant_summary.txt


In [8]:
# Cell 8 — Load and display saved outputs
with (OUTPUT_DIR / 'run_metadata.json').open() as f:
    saved = json.load(f)

print('=== Saved run_metadata.json ===')
for k, v in saved.items():
    print(f'  {k}: {v}')


=== Saved run_metadata.json ===
  run_type: uncoupled_notebook_regression
  timestamp: 2026-04-15T14:08:15.046694
  fcidump: /home/unfunnypanda/Proj_Flow/Fe4S4_251230orbital_-327.1920_10kdets/Fe4S4_251230orbital_-327.1920_10kdets/fcidump_cycle_6
  window_size: 15
  stride: 10
  trimci_config: DEFAULT_CONFIG
  total_dets: 118
  brute_force_dets: 10095
  fragment_n_dets: [51, 51, 16]
  fragment_energies: [-187.75020040863893, -219.71786393946374, -247.6589841523237]
  fragment_orbs: [[2, 3, 6, 7, 8, 10, 11, 21, 24, 25, 26, 27, 29, 32, 33], [4, 5, 7, 9, 10, 12, 13, 20, 21, 22, 23, 28, 31, 32, 33], [0, 1, 12, 13, 14, 15, 16, 17, 18, 19, 20, 22, 23, 30, 34, 35]]
  ratio: 0.011688954928182269
  savings_pct: 98.83110450718178
  regression_pass: True


## Interpretation

**Phase C (Path C) result:** 3 overlapping sliding-window fragments with `window_size=15`, `stride=10` on Fe4S4 (36 orbitals, 54 electrons).

| Metric | Value |
|---|---|
| Total determinants | 118 |
| Brute-force TrimCI reference | 10 095 |
| Determinant ratio | 0.012× |
| Savings | 98.8% |

**Key observations:**
- Fragment determinant counts `[51, 51, 16]` are stable across runs (regression-gated).
- Fragment energies vary by ≈1–2 Ha run-to-run due to stochastic heat-bath sampling (`num_runs=1` in `DEFAULT_CONFIG`). This is expected and not a bug.
- `max_final_dets='auto'` caps each fragment at ≈51 determinants for 15-orbital fragments. This is a truncation limit, not a physical result.
- Fragment energies are **not summable** — summing would double-count the two-electron interactions in overlapping orbital windows.
- The determinant count comparison (118 vs 10 095) is the scientific result: fragmenting reduces the CI space by 98.8%.

**Phase C is complete and the regression check passes.**

## How to run again

### From terminal
```bash
cd /home/unfunnypanda/Proj_Flow
source qflowenv/bin/activate

python -c "
from TrimCI_Flow.uncoupled import run_fragmented_trimci
from TrimCI_Flow.core import determinant_summary
r = run_fragmented_trimci(
    'Fe4S4_251230orbital_-327.1920_10kdets/Fe4S4_251230orbital_-327.1920_10kdets/fcidump_cycle_6',
    window_size=15, stride=10)
determinant_summary(r)
assert r.total_dets == 118
assert r.fragment_n_dets == [51, 51, 16]
print('Regression PASS')
"
```

### From this notebook
```bash
cd /home/unfunnypanda/Proj_Flow
source qflowenv/bin/activate
jupyter notebook TrimCI_Flow/Flow_Uncoupled.ipynb
# Run All Cells
```

### Expected output files
- `TrimCI_Flow/Outputs/uncoupled/outs_notebook_uncoupled/run_metadata.json`
- `TrimCI_Flow/Outputs/uncoupled/outs_notebook_uncoupled/fragment_results.json`
- `TrimCI_Flow/Outputs/uncoupled/outs_notebook_uncoupled/determinant_summary.txt`

### Regression assertion
`total_dets == 118` and `fragment_n_dets == [51, 51, 16]` must hold.
Fragment energies may vary run-to-run; this is expected with stochastic heat-bath selection.
